# Accessing your private data

This page explains how to access private MGnify data.

---

- Recommended: keep an `.env` file with `MG_USER` and `MG_PASSWORD` (see [.env.example](https://github.com/EBI-Metagenomics/mgnipy/blob/92a70c5f489d1fa16943585fcd50cef253bb61db/.env.example) in the repo). These are auto-loaded into `mgnipy.MGnipyConfig` via pydantic settings.
- Alternatively, pass credentials directly when creating a config: `config = MGnipyConfig(mg_user="...", mg_password="...")` and use that with `mgnipy.MGnipy` or resource proxies.

    <br>
    <details style="font-size:16px">
    <summary style="color:green; font-size:18px; font-weight:600">
        How the authentication works (sliding token):
    </summary>
    </summary>
    <h1></h1>

    - `mgnipy.MGnipyConfig` takes care of obtaining an authentication token from the [token_obtain_sliding](https://www.ebi.ac.uk/metagenomics/api/v2/#/Authentication/token_obtain_sliding) endpoint of the MGnify API using your username/password
    - The auth token is verified using the [token_verify](https://www.ebi.ac.uk/metagenomics/api/v2/#/Authentication/token_verify) endpoint and, if valid, refreshed using [token_refresh_sliding](https://www.ebi.ac.uk/metagenomics/api/v2/#/Authentication/token_refresh_sliding) when needed.
        - The high-level methods within `mgnipy.MGnipyConfig` involved are `obtain_auth_token`, `verify_auth_token`, `refresh_auth_token`, and `resolve_auth_token`.
    - The resolved token is stored in `MGnipyConfig.auth_token` for the session and used for authenticated API requests.
    - By default the token is cached on disk under a platform-appropriate cache dir (via `platformdirs`) in a file named auth_<hash>.json.
        - You can disable disk caching by setting `cache_dir=None` on MGnipyConfig.
    - On success `resolve_auth_token()` will confirm authentication, it prints `"Authenticated successfully."`

    <h1></h1>
    </details>
    <br>

---

## Quick configuration examples:

### **Option 1.** (Recommended) Auto-loading from an `.env` file
- Use an `.env` file (recommended). See [`.env.example`](https://github.com/EBI-Metagenomics/mgnipy/blob/92a70c5f489d1fa16943585fcd50cef253bb61db/.env.example) — variables `MG_USER` and `MG_PASSWORD`.
    - example `.env` contents:
        ```.env
                    MG_USER=<your MGnify or ENA username>
                    MG_PASSWORD=<your MGnify or ENA password>
        ```
- The .env file and `MG_USER` and `MG_PASSWORD` variables will auto be detected via pydantic settings and stored safely in `mgnipy.MGnipyConfig`

In [1]:
# if .env files then can just proceed as normal, for example in a notebook:
from mgnipy import MGnipy

MG = MGnipy()  # will automatically look for .env file and load credentials if found

DEBUG:numcodecs:Registering codec 'zlib'
DEBUG:numcodecs:Registering codec 'gzip'
DEBUG:numcodecs:Registering codec 'bz2'
DEBUG:numcodecs:Registering codec 'lzma'
DEBUG:numcodecs:Registering codec 'blosc'
DEBUG:numcodecs:Registering codec 'zstd'
DEBUG:numcodecs:Registering codec 'lz4'
DEBUG:numcodecs:Registering codec 'astype'
DEBUG:numcodecs:Registering codec 'delta'
DEBUG:numcodecs:Registering codec 'quantize'
DEBUG:numcodecs:Registering codec 'fixedscaleoffset'
DEBUG:numcodecs:Registering codec 'packbits'
DEBUG:numcodecs:Registering codec 'categorize'
DEBUG:numcodecs:Registering codec 'pickle'
DEBUG:numcodecs:Registering codec 'base64'
DEBUG:numcodecs:Registering codec 'shuffle'
DEBUG:numcodecs:Registering codec 'bitround'
DEBUG:numcodecs:Registering codec 'crc32'
DEBUG:numcodecs:Registering codec 'adler32'
DEBUG:numcodecs:Registering codec 'jenkins_lookup3'
DEBUG:numcodecs:Registering codec 'json2'
DEBUG:numcodecs:Registering codec 'vlen-utf8'
DEBUG:numcodecs:Registering codec 'vle

### **Option 1.5** If using a different filename than .env

Or if you prefer an env file name with a different filename then .env
1. you can manually load your given file by passing its path to `dotenv.load_dotenv`
2. and then use `os.getenv` to get out your MGnify user and pass variables
3. initiate `mgnipy.MGnipy` or resource-specific `MGnifier` instances (e.g., `mgnipy.V2.proxies`) with those login credentials like above

In [2]:
from dotenv import load_dotenv
import os
from mgnipy import MGnipyConfig

# load env variables from specific filename
load_dotenv("path/to/your-env-file")

# pass to config
config = MGnipyConfig(
    mg_user=os.getenv("MG_USER"), mg_password=os.getenv("MG_PASSWORD")
)

# pass config to MGnipy
MG = MGnipy(config=config)
# or directly to proxy
from mgnipy.V2.proxies import Biomes
biomes = Biomes(config=config)

DEBUG:mgnipy.V2.query_set:Initializing QuerySet for resource biomes
DEBUG:mgnipy._models.config:No valid auth token available, obtaining new one...
DEBUG:mgnipy._models.config:getting username and password...
DEBUG:mgnipy._models.config:retrieved username and password...
DEBUG:mgnipy._models.config:prepping body for token request...
DEBUG:mgnipy._models.config:requesting token from API...
DEBUG:httpcore.connection:connect_tcp.started host='www.ebi.ac.uk' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x119728ad0>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x11c961f40> server_hostname='www.ebi.ac.uk' timeout=None
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x11c83e250>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:sen


### **Option 2.** Explicity Configure
Manually pass the login credentials to `mgnipy.MGnipy` or resource-specific `MGnifier` instances (e.g., `mgnipy.V2.proxies`) at init.

In [3]:
# pass login credentials to config
config = MGnipyConfig(
    mg_user="this-is-a-fake-user-name", mg_password="this-is-a-fake-password"
)

# pass config to MGnipy
MG = MGnipy(config=config)

### **Option 3.** Pass credentials interactively
if no .env or not passed to MGnipy or proxies then when private endpoints called you will be prompted with an input window for user and then password. Such as for endpoints with only private data e.g. private_studies

In [4]:
# requires sliding authentication token using user and pass
my_studies = MG.private_studies

DEBUG:mgnipy.V2.query_set:Initializing QuerySet for resource private_studies
DEBUG:mgnipy._models.config:No valid auth token available, obtaining new one...
DEBUG:mgnipy._models.config:getting username and password...
DEBUG:mgnipy._models.config:Using configured MGnify credentials
DEBUG:mgnipy._models.config:retrieved username and password...
DEBUG:mgnipy._models.config:prepping body for token request...
DEBUG:mgnipy._models.config:requesting token from API...
DEBUG:httpcore.connection:connect_tcp.started host='www.ebi.ac.uk' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x1082fc210>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x11c962840> server_hostname='www.ebi.ac.uk' timeout=None
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x11c9d5990>
DEBUG:httpcore.http11:send_requ

## Then...

You can continue the same process as you would with non-private resources

In [5]:
# previewing query
# my_studies.explain()

In [6]:
# getting a page
# my_studies.get()